# Exercise 6: Segments per word across the family

*The Computational Phenotype · version 0.1.0*

> Take the same short text in Turkish, Kazakh, Uzbek and Tatar (for example a FLORES-200 paragraph). Segment 30 words per language by hand under the chapter's convention, then compare with the automatic output as in the Hands-On section. Report exact agreement, the gold and automatic means, and the words where analytic constructions change the word count.

FLORES-200 requires accepting its terms on Hugging Face, so the notebook uses the openly available **X-WMT** test sets (the same English news sentences translated into several Turkic languages) for Turkish, Kazakh and Uzbek. X-WMT has no Tatar; Bashkir (`ba`), a close relative, can be added instead, or Tatar from FLORES if you have access.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas requests`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import re
import requests
import pandas as pd
import turkicnlp
from turkicnlp.processors.morpheme_tokenizer import MorphemeTokenizer

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"
LANGS = {"tr": "tur", "kk": "kaz", "uz": "uzb"}          # X-WMT code -> TurkicNLP code

def xwmt(code):
    en = requests.get(f"{BASE}/en-{code}/en-{code}.en", timeout=30).text.splitlines()
    xx = requests.get(f"{BASE}/en-{code}/en-{code}.{code}", timeout=30).text.splitlines()
    return dict(zip(en, xx))

side = {c: xwmt(c) for c in LANGS}
common = [e for e in side["uz"] if all(e in side[c] for c in LANGS)]     # same English source sentence
words = lambda s: re.findall(r"\w+(?:['’ʻ]\w+)*", s)
print("aligned sentences:", len(common))

In [ ]:
N = 30
samples, rows = {}, []
for c, lang in LANGS.items():
    turkicnlp.download(lang, processors=["morph"])
    seg = MorphemeTokenizer(lang=lang); seg.load()
    ws, i = [], 0
    while len(ws) < N:
        ws += words(side[c][common[i]]); i += 1
    ws = ws[:N]
    results = [seg.segment(w) for w in ws]
    samples[c] = [(w, "-".join(m for m, _ in r.labeled), r.source) for w, r in zip(ws, results)]
    rows.append({"language": c, "sentences used": i, "automatic segments/word": round(sum(len(r.labeled) for r in results) / N, 2),
                 "words without an analysis": sum(r.source == "none" for r in results)})
pd.DataFrame(rows)

In [ ]:
# Word counts over all aligned sentences: the same content needs different numbers of words
pd.DataFrame([{"language": c, "words in the aligned sentences": sum(len(words(side[c][e])) for e in common)} for c in LANGS])

In [ ]:
# Hand segmentation. Fill in your reference segmentation for each word (stem-affix-affix ...),
# following the chapter's convention, then run the next cell. Leave "" for words you skip.
reference = {c: {w: "" for w, _, _ in samples[c]} for c in LANGS}
for c in LANGS:
    print(c, [w for w, _, _ in samples[c]])

In [ ]:
def agreement(c):
    pairs = [(reference[c][w], auto) for w, auto, _ in samples[c] if reference[c][w]]
    if not pairs:
        return None
    exact = sum(g == a for g, a in pairs)
    return {"language": c, "words checked": len(pairs), "exact": f"{exact}/{len(pairs)}",
            "gold segments/word": round(sum(len(g.split("-")) for g, _ in pairs) / len(pairs), 2),
            "automatic segments/word": round(sum(len(a.split("-")) for _, a in pairs) / len(pairs), 2)}

results = [r for r in (agreement(c) for c in LANGS) if r]
if results:
    display(pd.DataFrame(results))
else:
    print("Fill in `reference` first.")

## Worked solution (run on 18 October 2026)

On the first aligned X-WMT sentences, 30 words per language, the automatic segmenter (Apertium only) produced:

| Language | Sentences used | Automatic segments/word | Words without an analysis |
|---|---|---|---|
| Turkish | 3 | 1.93 | 5 |
| Kazakh | 3 | 1.57 | 6 |
| Uzbek | 2 | 1.70 | 5 |

Over all 500 aligned sentences, the Turkish translations contain 9,019 words, the Kazakh 8,664 and the Uzbek 9,617.

**Interpretation.** These are not language properties. The means depend on the segmenter's coverage (5–6 of 30 words had no analysis and count as one segment), on its errors (for example Turkish *sudan* 'from the water' left unsegmented), and on the translations, which were produced independently and differ in wording. The word counts differ for the same content because of analytic constructions, compounds and translation choices. Exact agreement and the gold means can only be reported after you have segmented the words by hand (cells above); a reference produced by the same segmenter would only measure its agreement with itself.